**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/07-paper-bug-hunt.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/bug-hunt) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Cursor](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7G · Bug Hunt 🐞: Eight Classic Paper-Implementation Bugs (pairs)

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| 45 min, in pairs | Google Colab (CPU is enough, GPU is faster) or Cursor / VS Code on your laptop | the debugging instinct: eight planted bugs from real student and research code, each one runs without an error message but gives wrong results |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. **Play in pairs:** one person reads the paper/slide, the other edits the code. Swap after every bug. First pair to 8/8 shows the scoreboard to the teacher.

The nastiest bugs in deep learning **do not crash**. The code runs, the loss goes down a bit, and the model is just…
worse. Each cell below contains one such bug, copied from real-world code. Fix it, then run the check underneath.
Rules: change as **little** as possible (usually one line), and say in one sentence what was wrong.

| Bug | Paper | Where to look |
|---|---|---|
| 1, 2 | Attention Is All You Need, Eq. 1 | scaled dot-product attention |
| 3 | §3.1 decoder | the causal mask |
| 4 | §3.2.2 | splitting heads |
| 5 | U-Net Fig. 1 | the skip connection |
| 6 | ResNet Fig. 2 | the residual block |
| 7 | ViT Eq. 1 | patches |
| 8 | any paper | the training loop |

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

## Bug 1 · Attention scores
<details><summary><b>Hint 1</b></summary>

Read Eq. 1 aloud: softmax of Q Kᵀ divided by *what*?

</details>

In [ ]:
def attention_1(q, k, v):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / d_k
    return scores.softmax(dim=-1) @ v

In [ ]:
C.check_bug1(attention_1)

## Bug 2 · Attention weights
<details><summary><b>Hint 1</b></summary>

`scores` is (B, L_q, L_k). Each query's weights over the keys must sum to 1. Print `w.sum(-1)` and `w.sum(-2)`.

</details>

In [ ]:
def attention_2(q, k, v):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    w = scores.softmax(dim=1)
    return w @ v

In [ ]:
C.check_bug2(attention_2)

## Bug 3 · The causal mask
<details><summary><b>Hint 1</b></summary>

`print(causal_mask_3(4).int())`. Token 0 should be allowed to see token 0. What does `diagonal=` do in `torch.tril`?

</details>

In [ ]:
def causal_mask_3(L):
    return torch.tril(torch.ones(L, L, dtype=torch.bool), diagonal=-1)

In [ ]:
C.check_bug3(causal_mask_3)

## Bug 4 · Splitting the heads
<details><summary><b>Hint 1</b></summary>

The shape after `.view(B, self.h, L, self.d_k)` is right, but which numbers end up in head 0? Lab 7B §6 shows the correct two-step recipe.

</details>

In [ ]:
class MHA_4(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        self.h, self.d_k = h, d_model // h
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, L, D = x.shape
        q = self.w_q(x).view(B, self.h, L, self.d_k)
        k = self.w_k(x).view(B, self.h, L, self.d_k)
        v = self.w_v(x).view(B, self.h, L, self.d_k)
        w = (q @ k.transpose(-2, -1) / math.sqrt(self.d_k)).softmax(-1)
        out = (w @ v).reshape(B, L, D)
        return self.w_o(out)

In [ ]:
C.check_bug4(MHA_4)

## Bug 5 · U-Net up-step
<details><summary><b>Hint 1</b></summary>

Run it and read the error, or print `x.shape` and `skip.shape` before `torch.cat`. Which axis are the channels?

</details>

In [ ]:
class Up_5(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.up = nn.ConvTranspose2d(cin, cout, 2, stride=2)
        self.conv = nn.Sequential(nn.Conv2d(2 * cout, cout, 3, padding=1), nn.ReLU())

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([skip, x], dim=0)
        return self.conv(x)

In [ ]:
C.check_bug5(Up_5)

## Bug 6 · Residual block (two bugs in one block!)
<details><summary><b>Hint 1</b></summary>

BasicBlock(32, 64, stride=2) changes channels AND size: `x` cannot be added as it is.

</details>
<details><summary><b>Hint 2</b></summary>

Figure 2 of the ResNet paper: where is the last ReLU?

</details>

In [ ]:
class Block_6(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.bn2 = nn.BatchNorm2d(cout)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = torch.relu(self.bn2(self.conv2(out)))
        return out + x

In [ ]:
C.check_bug6(Block_6)

## Bug 7 · Patches
<details><summary><b>Hint 1</b></summary>

`x = torch.arange(32.).view(1, 2, 4, 4)`; compare `patchify_7(x, 2)[0, 0]` with `x[0, :, :2, :2]`.

</details>
<details><summary><b>Hint 2</b></summary>

Lab 7F, Your turn 1: split → permute → reshape. Patch layout must be (P, P, C).

</details>

In [ ]:
def patchify_7(x, P):
    B, C, H, W = x.shape
    return x.reshape(B, (H // P) * (W // P), P * P * C)

In [ ]:
C.check_bug7(patchify_7)

## Bug 8 · The training step
<details><summary><b>Hint 1</b></summary>

Lab 7A §5: the five lines. One is missing. What does PyTorch do with `.grad` on every `backward()`?

</details>

In [ ]:
def train_step_8(model, opt, X, y):
    loss = F.mse_loss(model(X), y)
    loss.backward()
    opt.step()
    return loss.item()

In [ ]:
C.check_bug8(train_step_8)

## 🏆 Scoreboard

In [ ]:
C.scoreboard()

## ✍️ Reflection (one line each)
For every bug write: **what was wrong · how you found it · which habit would have caught it earlier**
(e.g. "compare with the PyTorch reference", "print shapes", "test with a tiny example you can check by hand",
"overfit one batch first").

Those habits are your **debugging checklist** for the Paper Implementation Challenge (Lab 7H).